In [ ]:
#load data

import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import Normalizer
from sklearn.linear_model import LogisticRegression
from sklearn import preprocessing
from torch.utils.data import Dataset, DataLoader
from torch.nn.modules.loss import _Loss
from torch import Tensor
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

data = pd.read_csv(".../compas-scores-two-years.csv")
data = pd.DataFrame(data)

data_w = data[data['race'] == 'Caucasian']
data_b = data[data['race'] == 'African-American']

data = pd.concat([data_w, data_b])

for col in set(data.columns) - set(data.describe().columns):
  data[col] = data[col].astype('category')


def oneHotCatVars(df, df_cols):

    df_1 = df.drop(columns = df_cols, axis = 1)
    df_2 = pd.get_dummies(df[df_cols])

    return (pd.concat([df_1, df_2], axis=1, join='inner'))

data_preprocessed = oneHotCatVars(data, data.select_dtypes('category').columns)


normalize_columns = ['age', 'juv_fel_count', 'juv_misd_count', 'juv_other_count', 'priors_count']

def normalize(columns):
  scaler = preprocessing.StandardScaler()
  data_preprocessed[columns] = scaler.fit_transform(data_preprocessed[columns])


class ShelterOutcomeDataset(Dataset):
    def __init__(self, X, Y, A):
        self.x = X.to_numpy().astype(np.float32)
        self.y = Y
        self.a =  A

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx], self.a[idx]

normalize(normalize_columns)

x_train, x_test = train_test_split(data_preprocessed)


b_test = x_test[x_test['race_African-American'] == 1]
w_test = x_test[x_test['race_Caucasian'] == 1]

train_x = x_train.drop(['two_year_recid'],axis=1)
train_x = train_x.drop(['race_African-American'],axis=1)
train_x = train_x.drop(['race_Caucasian'],axis=1)
train_label = x_train['two_year_recid'].to_numpy()
train_sen = x_train['race_African-American'].to_numpy()
test_x = x_test.drop(['two_year_recid'],axis=1)
test_x = test_x.drop(['race_African-American'],axis=1)
test_x = test_x.drop(['race_Caucasian'],axis=1)
test_label = x_test['two_year_recid'].to_numpy()
test_sen = x_test['race_African-American'].to_numpy()

#baseline model

class Classifier_complx(nn.Module):
    def __init__(self, input_size = 422, hidden_size = 512):
        super(Classifier_complx, self).__init__()

        self.dense1 = nn.Linear(input_size, hidden_size)
        self.dense2 = nn.Linear(hidden_size, hidden_size)
        self.dense3 = nn.Linear(hidden_size, 2)

    def forward(self, x):
        x = F.relu(self.dense1(x))
        x1 = F.relu(self.dense2(x))
        x = self.dense3(x1)

        return F.log_softmax(x,-1)


class ShelterDataset(Dataset):
    def __init__(self, X, Y, A, X1, Y1, A1):
        self.x = np.vstack((np.array(X).squeeze().astype(np.float32), X1.to_numpy().astype(np.float32)))
        self.y = np.hstack((np.array(Y).squeeze(), Y1))
        self.a = np.hstack((np.array(A).squeeze(), A1))

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx], self.a[idx]

In [ ]:
train_ds = ShelterOutcomeDataset(train_x, train_label, train_sen)
test_ds = ShelterOutcomeDataset(test_x, test_label, test_sen)


batch_size_train, batch_size_test = 256, test_x.shape[0]
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=batch_size_train, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=batch_size_test, shuffle=False)


classifier = Classifier_complx()
optimizer = optim.SGD(classifier.parameters(), lr=0.01, momentum=0.8)

def pgd_attack(model, images, labels, eps, alpha=2/255, iters=40) :
    loss = nn.CrossEntropyLoss()

    ori_images = images.data

    for i in range(iters) :
        images.requires_grad = True
        outputs = model(images)

        model.zero_grad()
        cost = loss(outputs, labels)
        cost.backward()

        adv_images = images + alpha*images.grad.sign()
        eta = torch.clamp(adv_images - ori_images, min=-eps, max=eps)
        images = torch.clamp(ori_images + eta, min=0, max=1).detach_()

    return images

def gradient(y, x, grad_outputs=None):
    if grad_outputs is None:
        grad_outputs = torch.ones_like(y)
    grad = torch.autograd.grad(y, [x], grad_outputs = grad_outputs, create_graph=True)[0]
    return grad


def train(epoch):

  classifier.train()

  for batch_idx, (x, y, a) in enumerate(train_loader):
    x.requires_grad = True
    optimizer.zero_grad()
    idx_bp = np.where(np.logical_and(a==1,y==1))
    idx_wp = np.where(np.logical_and(a==0,y==1))
    idx_bn = np.where(np.logical_and(a==1,y==0))
    idx_wn = np.where(np.logical_and(a==0,y==0))
    output = classifier(x)
    bp = output[idx_bp] / len(np.array(idx_bp))
    wp = output[idx_wp] / len(np.array(idx_wp))
    bn = output[idx_bn] / len(np.array(idx_bn))
    wn = output[idx_wn] / len(np.array(idx_wn))
    eod = abs(torch.sum(torch.exp(bn)[:,0]) / len(np.array(idx_bn)[0]) - torch.sum(torch.exp(wn)[:,0]) / len(np.array(idx_wn)[0])) + abs(torch.sum(torch.exp(bp)[:,1]) / len(np.array(idx_bp)[0]) - torch.sum(torch.exp(wp)[:,1]) / len(np.array(idx_wp)[0]))
    loss = F.nll_loss(output, y) + 0.8 * eod
    x_adv = pgd_attack(classifier, x, y, 0.05)
    output_adv  = classifier(x_adv)
    bp_adv = output_adv[idx_bp] / len(np.array(idx_bp))
    wp_adv = output_adv[idx_wp] / len(np.array(idx_wp))
    bn_adv = output_adv[idx_bn] / len(np.array(idx_bn))
    wn_adv = output_adv[idx_wn] / len(np.array(idx_wn))
    eod = abs(torch.sum(torch.exp(bn_adv)[:,0]) / len(np.array(idx_bn)[0]) - torch.sum(torch.exp(wn_adv)[:,0]) / len(np.array(idx_wn)[0])) + abs(torch.sum(torch.exp(bp_adv)[:,1]) / len(np.array(idx_bp)[0]) - torch.sum(torch.exp(wp_adv)[:,1]) / len(np.array(idx_wp)[0]))
    loss_adv = F.nll_loss(output_adv, y) + 0.15 * eod
    loss.backward()
    optimizer.step()

    if batch_idx % 20 == 0:
      train_losses.append(loss.item())
      train_counter.append(
        (batch_idx*256) + ((epoch-1)*len(train_loader.dataset)))
    # if batch_idx % 20 == 0:
    #   print(f'Epoch {epoch}: [{batch_idx*len(x)}/{len(train_loader.dataset)}] Loss: {loss.item()}')



def test(epoch):

  classifier.eval()
  test_loss = 0
  correct = 0

  with torch.no_grad():
    for x, y, a in test_loader:
      y = y.long()
      output = classifier(x)
      test_loss += F.nll_loss(output, y).item()
      pred = output.data.max(1, keepdim=True)[1]
      correct += pred.eq(y.data.view_as(pred)).sum()
  test_losses.append(test_loss)
  test_counter.append(len(train_loader.dataset)*epoch)

  print(f'Test result of FC on epoch {epoch}: Avg loss is {test_loss}, Accuracy: {100.*correct/len(test_loader.dataset)}%')

train_losses = []
train_counter = []
test_losses = []
test_counter = []
max_epoch = 60


for epoch in range(1, max_epoch+1):
  train(epoch)
  test(epoch)

def model_eval(actual, pred):

    confusion = pd.crosstab(actual, pred, rownames=['Actual'], colnames=['Predicted'])
    try:
      TP = confusion.loc[1,1] + 1
    except KeyError:
      TP = 1
    try:
      TN = confusion.loc[0,0] + 1
    except KeyError:
      TN = 1
    try:
      FP = confusion.loc[0,1] + 1
    except KeyError:
      FP = 1
    try:
      FN = confusion.loc[1,0] + 1
    except KeyError:
      FN = 1


    out = {}
    out['ALL'] = (TP+TN+FP+FN-4)
    out['DP'] = (TP+FP-2)/(TP+TN+FP+FN-4)
    out['TPR'] =  (TP-1)/(TP+FN-2)
    out['TNR'] = (TN-1)/(FP+TN-2)
    out['FPR'] = (FP-1)/(FP+TN-2)
    out['FNR'] = (FN-1)/(TP+FN-2)
    out['ACR'] = (TP+TN-2)/(TP+TN+FP+FN-4)

    return out

for x, y, a in test_loader:
      y = y.long()
      output = classifier(x)
      idx_b = np.where(a==1)
      y_b = y[[idx_b]]
      pred_b = output[[idx_b]]
      pred_b = torch.squeeze(pred_b,0).data.max(1, keepdim=True)[1]
      idx_w = np.where(a==0)
      y_w = y[[idx_w]]
      pred_w = output[[idx_w]]
      pred_w = torch.squeeze(pred_w,0).data.max(1, keepdim=True)[1]


w = model_eval(torch.squeeze(y_w,0), pred_w.detach().numpy().reshape(pred_w.shape[0]))
b = model_eval(torch.squeeze(y_b,0), pred_b.detach().numpy().reshape(pred_b.shape[0]))



DI1 = 100 * abs(w['DP'] - b['DP'])
DFPR1 = 100 * abs(w['TNR'] - b['TNR'])
DFNR1 = 100 * abs(w['TPR'] - b['TPR'])
w_TNR  = 100 * w['TNR']
w_TPR  = 100 * w['TPR']
b_TNR  = 100 * b['TNR']
b_TPR  = 100 * b['TPR']


print(f'Race: disparate impact is {DI1}%, eod is {DFPR1+DFNR1}%, w_TNR is {w_TNR}%, w_TPR is {w_TPR}%, b_TNR is {b_TNR}%, b_TPR is {b_TPR}%.')

Test result of FC on epoch 1: Avg loss is 0.689518392086029, Accuracy: 53.51105499267578%
Test result of FC on epoch 2: Avg loss is 0.6896576881408691, Accuracy: 53.51105499267578%
Test result of FC on epoch 3: Avg loss is 0.6888607740402222, Accuracy: 53.51105499267578%
Test result of FC on epoch 4: Avg loss is 0.6881395578384399, Accuracy: 53.51105499267578%
Test result of FC on epoch 5: Avg loss is 0.687667727470398, Accuracy: 53.51105499267578%
Test result of FC on epoch 6: Avg loss is 0.687399685382843, Accuracy: 53.51105499267578%
Test result of FC on epoch 7: Avg loss is 0.6873295307159424, Accuracy: 53.4460334777832%
Test result of FC on epoch 8: Avg loss is 0.6863663792610168, Accuracy: 53.38101577758789%
Test result of FC on epoch 9: Avg loss is 0.6858991980552673, Accuracy: 53.38101577758789%
Test result of FC on epoch 10: Avg loss is 0.685292661190033, Accuracy: 53.64109420776367%
Test result of FC on epoch 11: Avg loss is 0.6844018697738647, Accuracy: 53.7711296081543%
Tes

In [ ]:
# FGSM attack code

def pgd_attack(model, images, labels, eps, alpha=2/255, iters=40) :
    loss = nn.CrossEntropyLoss()

    ori_images = images.data

    for i in range(iters) :
        images.requires_grad = True
        outputs = model(images)

        model.zero_grad()
        cost = loss(outputs, labels)
        cost.backward()

        adv_images = images + alpha*images.grad.sign()
        eta = torch.clamp(adv_images - ori_images, min=-eps, max=eps)
        images = torch.clamp(ori_images + eta, min=0, max=1).detach_()

    return images


def test( model, test_loader, epsilon):

    # Accuracy counter
    correct = 0
    adv_examples = []
    model.eval()

    # Loop over all examples in test set
    for data, target, sen in test_loader:

        # Set requires_grad attribute of tensor. Important for Attack
        data.requires_grad = True

        # Forward pass the data through the model
        idx_b = np.where(sen==1)
        idx_w = np.where(sen==0)
        idx_bp = np.where(np.logical_and(sen==1,target==1))
        idx_wp = np.where(np.logical_and(sen==0,target==1))
        idx_bn = np.where(np.logical_and(sen==1,target==0))
        idx_wn = np.where(np.logical_and(sen==0,target==0))
        output0 = model(data)
        init_pred = torch.squeeze(output0.max(1, keepdim=True)[1],1) # get the index of the max log-probability

        # If the initial prediction is wrong, dont bother attacking, just move on
        #for i in range(len(init_pred)):
          #if init_pred[i].item() != target[i].item():
              #\continue

        # Calculate the loss
        p1 = torch.squeeze(output0[[idx_bp]],0)
        p2 = torch.squeeze(output0[[idx_wp]],0)
        p3 = torch.squeeze(output0[[idx_bn]],0)
        p4 = torch.squeeze(output0[[idx_wn]],0)
        p5 = torch.squeeze(output0[[idx_b]],0)
        p6 = torch.squeeze(output0[[idx_w]],0)
        loss = abs(torch.sum(torch.exp(p3)[:,0]) / len(np.array(idx_bn)[0]) - torch.sum(torch.exp(p4)[:,0]) / len(np.array(idx_wn)[0])) + abs(torch.sum(torch.exp(p1)[:,1]) / len(np.array(idx_bp)[0]) - torch.sum(torch.exp(p2)[:,1]) / len(np.array(idx_wp)[0]))
#F.nll_loss(output, target) - #abs(torch.sum(torch.exp(p5)[:,0]) / len(np.array(idx_b)[0])-torch.sum(torch.exp(p6)[:,0]) / len(np.array(idx_w)[0]))
        # Zero all existing gradients
        model.zero_grad()

        # Calculate gradients of model in backward pass
        loss.backward()

        # Collect datagrad
        data_grad = data.grad.data

        # Call FGSM Attack
        perturbed_data = fgsm_attack(data, epsilon, data_grad)

        # Re-classify the perturbed image
        output = model(perturbed_data)

        # Check for success
        final_pred = output.max(1, keepdim=True)[1] # get the index of the max log-probability
        y_b = target[[idx_b]]
        pred_b_soft = output[[idx_b]]
        pred_b = torch.squeeze(pred_b_soft,0).data.max(1, keepdim=True)[1]
        y_w = target[[idx_w]]
        pred_w_soft = output[[idx_w]]
        pred_w = torch.squeeze(pred_w_soft,0).data.max(1, keepdim=True)[1]
        w = model_eval(torch.squeeze(y_w,0), pred_w.detach().numpy().reshape(pred_w.shape[0]))
        b = model_eval(torch.squeeze(y_b,0), pred_b.detach().numpy().reshape(pred_b.shape[0]))
        DI1 = 100 * abs(w['DP'] - b['DP'])
        DFPR1 = 100 * abs(w['TNR'] - b['TNR'])
        DFNR1 = 100 * abs(w['TPR'] - b['TPR'])
        w_TNR  = 100 * w['TNR']
        w_TPR  = 100 * w['TPR']
        b_TNR  = 100 * b['TNR']
        b_TPR  = 100 * b['TPR']
        DI1 = 100 * abs(w['DP'] - b['DP'])
        DFPR1 = 100 * abs(w['TNR'] - b['TNR'])
        DFNR1 = 100 * abs(w['TPR'] - b['TPR'])
        for i in range(len(final_pred)):
          if final_pred[i].item() == target[i].item():
              correct += 1
              # Special case for saving 0 epsilon examples
              if (epsilon == 0) and (len(adv_examples) < 5):
                  adv_ex = perturbed_data[i].squeeze().detach().cpu().numpy()
                  adv_examples.append( (init_pred[i].item(), final_pred[i].item(), adv_ex) )
          else:
              # Save some adv examples for visualization later
              if len(adv_examples) < 5:
                  adv_ex = perturbed_data[i].squeeze().detach().cpu().numpy()
                  adv_examples.append( (init_pred[i].item(), final_pred[i].item(), adv_ex) )

    # Calculate final accuracy for this epsilon
    final_acc = correct/data.shape[0]
    print("Epsilon: {}\tTest Accuracy = {} / {} = {}\tDisparate impact = {}%\tEod = {}%.')".format(epsilon, correct, data.shape[0], final_acc, DI1, DFPR1+DFNR1))

    # Return the accuracy and an adversarial example
    return DI1, DFPR1+DFNR1, final_acc, adv_examples, final_pred, w_TNR, w_TPR, b_TNR, b_TPR, pred_w_soft.detach().numpy(), pred_b_soft.detach().numpy(), output.detach().numpy()

In [ ]:
accuracies_advf_in = []
EOd_advf_in = []
DI_advf_in = []
w_tpr_advf_in = []
w_tnr_advf_in = []
b_tpr_advf_in = []
b_tnr_advf_in = []

epsilons = [0, .01, .05, .1, .15, .2, .25, .3, .35, .4, .45, .5]

# Run test for each epsilon
for eps in epsilons:
    di, eod, acc, ex, fp, w1, w2, b1, b2, pw, pb, pred = test(classifier, test_loader, eps)
    accuracies_advf_in.append(acc)
    EOd_advf_in.append(eod / 100)
    DI_advf_in.append(di / 100)
    w_tnr_advf_in.append(w1 / 100)
    w_tpr_advf_in.append(w2 / 100)
    b_tnr_advf_in.append(b1 / 100)
    b_tpr_advf_in.append(b2 / 100)

Epsilon: 0	Test Accuracy = 964 / 1538 = 0.6267880364109233	Disparate impact = 13.311973147194573%	Eod = 23.988309871447136%.')
Epsilon: 0.01	Test Accuracy = 936 / 1538 = 0.6085825747724317	Disparate impact = 30.163271014627007%	Eod = 58.35260072906121%.')
Epsilon: 0.05	Test Accuracy = 924 / 1538 = 0.6007802340702211	Disparate impact = 48.773151553790065%	Eod = 97.90115543441118%.')
Epsilon: 0.1	Test Accuracy = 933 / 1538 = 0.606631989596879	Disparate impact = 57.157569515962926%	Eod = 114.61931475807752%.')
Epsilon: 0.15	Test Accuracy = 938 / 1538 = 0.6098829648894668	Disparate impact = 61.38002059732235%	Eod = 123.08214585791032%.')
Epsilon: 0.2	Test Accuracy = 938 / 1538 = 0.6098829648894668	Disparate impact = 64.26364572605561%	Eod = 128.85077274308944%.')
Epsilon: 0.25	Test Accuracy = 932 / 1538 = 0.6059817945383615	Disparate impact = 69.20700308959835%	Eod = 138.72075161041187%.')
Epsilon: 0.3	Test Accuracy = 910 / 1538 = 0.5916775032509753	Disparate impact = 74.97425334706487%	Eo

In [ ]:
accuracies_adv = []
EOd_adv = []
DI_adv = []
w_tpr_adv = []
w_tnr_adv = []
b_tpr_adv = []
b_tnr_adv = []

epsilons = [0, .01, .05, .1, .15, .2, .25, .3, .35, .4, .45, .5]

# Run test for each epsilon
for eps in epsilons:
    di, eod, acc, ex, fp, w1, w2, b1, b2, pw, pb, pred = test(classifier, test_loader, eps)
    accuracies_adv.append(acc)
    EOd_adv.append(eod)
    DI_adv.append(di)
    w_tnr_adv.append(w1 / 100)
    w_tpr_adv.append(w2 / 100)
    b_tnr_adv.append(b1 / 100)
    b_tpr_adv.append(b2 / 100)

Epsilon: 0	Test Accuracy = 915 / 1538 = 0.5949284785435631	Disparate impact = 13.853166377596349%	Eod = 25.272417680070696%.')
Epsilon: 0.01	Test Accuracy = 805 / 1538 = 0.523407022106632	Disparate impact = 16.29933584330978%	Eod = 33.494676603376035%.')
Epsilon: 0.05	Test Accuracy = 619 / 1538 = 0.4024707412223667	Disparate impact = 9.114528251336068%	Eod = 22.060802628457168%.')
Epsilon: 0.1	Test Accuracy = 349 / 1538 = 0.2269180754226268	Disparate impact = 2.2279566253507577%	Eod = 14.23695987092896%.')
Epsilon: 0.15	Test Accuracy = 246 / 1538 = 0.1599479843953186	Disparate impact = 0.7431222414935768%	Eod = 13.351371524538159%.')
Epsilon: 0.2	Test Accuracy = 139 / 1538 = 0.09037711313394019	Disparate impact = 3.400101527137356%	Eod = 8.105523418548236%.')
Epsilon: 0.25	Test Accuracy = 73 / 1538 = 0.04746423927178153	Disparate impact = 4.118899558638972%	Eod = 8.16500279902967%.')
Epsilon: 0.3	Test Accuracy = 55 / 1538 = 0.03576072821846554	Disparate impact = 4.610671630215607%	Eod 

In [ ]:
accuracies_adv1 = []
EOd_adv1 = []
DI_adv1 = []
w_tpr_adv1 = []
w_tnr_adv1 = []
b_tpr_adv1 = []
b_tnr_adv1 = []

epsilons = [0, .01, .05, .1, .15, .2, .25, .3, .35, .4, .45, .5]

# Run test for each epsilon
for eps in epsilons:
    di, eod, acc, ex, fp, w1, w2, b1, b2, pw, pb, pred = test(classifier, test_loader, eps)
    accuracies_adv1.append(acc)
    EOd_adv1.append(eod / 100)
    DI_adv1.append(di / 100)
    w_tnr_adv1.append(w1 / 100)
    w_tpr_adv1.append(w2 / 100)
    b_tnr_adv1.append(b1 / 100)
    b_tpr_adv1.append(b2 / 100)

Epsilon: 0	Test Accuracy = 943 / 1538 = 0.6131339401820546	Disparate impact = 11.688436738988475%	Eod = 19.983848719895732%.')
Epsilon: 0.01	Test Accuracy = 819 / 1538 = 0.5325097529258778	Disparate impact = 8.594232812847407%	Eod = 17.063275592528527%.')
Epsilon: 0.05	Test Accuracy = 532 / 1538 = 0.3459037711313394	Disparate impact = 4.780662363422778%	Eod = 18.355906771715993%.')
Epsilon: 0.1	Test Accuracy = 396 / 1538 = 0.2574772431729519	Disparate impact = 5.086686379655355%	Eod = 23.005527596732943%.')
Epsilon: 0.15	Test Accuracy = 314 / 1538 = 0.20416124837451236	Disparate impact = 1.432605563404571%	Eod = 19.049583367708934%.')
Epsilon: 0.2	Test Accuracy = 284 / 1538 = 0.1846553966189857	Disparate impact = 0.4175442097910209%	Eod = 18.183318208068645%.')
Epsilon: 0.25	Test Accuracy = 251 / 1538 = 0.16319895968790638	Disparate impact = 0.8067269260867926%	Eod = 17.035958843565478%.')
Epsilon: 0.3	Test Accuracy = 231 / 1538 = 0.15019505851755527	Disparate impact = 1.30392841699098

In [ ]:
plt.figure(figsize=(5,5))
plt.plot(epsilons, accuracies, label='baseline', color='b', marker='*')
plt.plot(epsilons, accuracies_adv1, label='adversarial training', color='y', marker='*')
plt.yticks(np.arange(0, 1.1, step=0.1))
plt.xticks(np.arange(0, .55, step=0.05))
plt.title("Epsilon vs accuracies")
plt.xlabel("Epsilon")
plt.ylabel("accuracies")
plt.legend(frameon=False, loc = 'best')
plt.show()

In [ ]:
accuracies = []
examples = []
DI = []
EOd = []
w_tpr = []
w_tnr = []
b_tpr = []
b_tnr = []
p_w = []
p_b = []
p = []
epsilons = [0, .01, .05, .1, .15, .2, .25, .3, .35, .4, .45, .5]

# Run test for each epsilon
for eps in epsilons:
    di, eod, acc, ex, fp, w1, w2, b1, b2, pw, pb, pred = test(classifier, test_loader, eps)
    accuracies.append(acc)
    examples.append(ex)
    DI.append(di / 100)
    EOd.append(eod / 100)
    w_tnr.append(w1 / 100)
    w_tpr.append(w2 / 100)
    b_tnr.append(b1 / 100)
    b_tpr.append(b2 / 100)
    p_w.append(pw)
    p_b.append(pb)
    p.append(pred)

Epsilon: 0	Test Accuracy = 967 / 1538 = 0.628738621586476	Disparate impact = 21.4479602916085%	Eod = 40.0721221414784%.')
Epsilon: 0.01	Test Accuracy = 712 / 1538 = 0.46293888166449937	Disparate impact = 20.139740823779906%	Eod = 42.27754795255787%.')
Epsilon: 0.05	Test Accuracy = 108 / 1538 = 0.07022106631989597	Disparate impact = 2.760269046913999%	Eod = 9.91770768801143%.')
Epsilon: 0.1	Test Accuracy = 2 / 1538 = 0.0013003901170351106	Disparate impact = 9.176220088272213%	Eod = 0.5434782608695652%.')
Epsilon: 0.15	Test Accuracy = 0 / 1538 = 0.0	Disparate impact = 9.501952987295015%	Eod = 0.0%.')
Epsilon: 0.2	Test Accuracy = 0 / 1538 = 0.0	Disparate impact = 9.501952987295015%	Eod = 0.0%.')
Epsilon: 0.25	Test Accuracy = 0 / 1538 = 0.0	Disparate impact = 9.501952987295015%	Eod = 0.0%.')
Epsilon: 0.3	Test Accuracy = 0 / 1538 = 0.0	Disparate impact = 9.501952987295015%	Eod = 0.0%.')
Epsilon: 0.35	Test Accuracy = 0 / 1538 = 0.0	Disparate impact = 9.501952987295015%	Eod = 0.0%.')
Epsilon:

In [ ]:
accuracies_f_in = []
examples_f_in = []
DI_f_in = []
EOd_f_in = []
w_tpr_f_in = []
w_tnr_f_in = []
b_tpr_f_in = []
b_tnr_f_in = []
p_w_f_in = []
p_b_f_in = []
epsilons = [0, .01, .05, .1, .15, .2, .25, .3, .35, .4, .45, .5]

# Run test for each epsilon
for eps in epsilons:
    di, eod, acc, ex, fp, w1, w2, b1, b2, pw, pb, pred = test(classifier, test_loader, eps)
    accuracies_f_in.append(acc)
    examples_f_in.append(ex)
    DI_f_in.append(di / 100)
    EOd_f_in.append(eod / 100)
    w_tnr_f_in.append(w1 / 100)
    w_tpr_f_in.append(w2 / 100)
    b_tnr_f_in.append(b1 / 100)
    b_tpr_f_in.append(b2 / 100)
    p_w_f_in.append(pw)
    p_b_f_in.append(pb)

Epsilon: 0	Test Accuracy = 998 / 1538 = 0.6488946684005201	Disparate impact = 11.794354838709676%	Eod = 16.497663482048324%.')
Epsilon: 0.01	Test Accuracy = 996 / 1538 = 0.647594278283485	Disparate impact = 55.40747028862478%	Eod = 107.06166194489339%.')
Epsilon: 0.05	Test Accuracy = 884 / 1538 = 0.5747724317295189	Disparate impact = 100.0%	Eod = 200.0%.')
Epsilon: 0.1	Test Accuracy = 884 / 1538 = 0.5747724317295189	Disparate impact = 100.0%	Eod = 200.0%.')
Epsilon: 0.15	Test Accuracy = 884 / 1538 = 0.5747724317295189	Disparate impact = 100.0%	Eod = 200.0%.')
Epsilon: 0.2	Test Accuracy = 884 / 1538 = 0.5747724317295189	Disparate impact = 100.0%	Eod = 200.0%.')
Epsilon: 0.25	Test Accuracy = 884 / 1538 = 0.5747724317295189	Disparate impact = 100.0%	Eod = 200.0%.')
Epsilon: 0.3	Test Accuracy = 884 / 1538 = 0.5747724317295189	Disparate impact = 100.0%	Eod = 200.0%.')
Epsilon: 0.35	Test Accuracy = 884 / 1538 = 0.5747724317295189	Disparate impact = 100.0%	Eod = 200.0%.')
Epsilon: 0.4	Test Ac